## E-Commerce Data Pipeline

## Technologies

- Databricks Free Edition
- PySpark
- Delta Lake
- Hive Metastore

## Architecture

Landing → Bronze → Silver → Gold → Reconciliation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
landing_path = "/Volumes/workspace/default/ecommerce_landing_files/"
orders_file = landing_path + "orders.csv"
order_items_file = landing_path + "order_items.csv"
customers_file = landing_path + "customers.csv"
inventory_file = landing_path + "inventory.csv"

In [0]:
# Create Landing Database
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_landing")
# Create Bronze Database
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_bronze")
# Create Silver Database
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_silver")
# Create Gold Database
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_gold")
# Show all databases
display(spark.sql("SHOW DATABASES"))

databaseName
default
ecommerce_bronze
ecommerce_gold
ecommerce_landing
ecommerce_silver
information_schema


### 01 - Landing Layer

### Objective

The Landing layer stores the raw CSV files exactly as they arrive.

No cleaning.

No datatype conversion.

Only metadata is added.

All columns remain STRING.

In [0]:
# Read orders.csv
orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(orders_file)
)
orders_df.show(5)

+----------+-----------+-------------------+---------+------------+-------+--------+---------+
|  order_id|customer_id|         order_date|   status|total_amount| region|category|warehouse|
+----------+-----------+-------------------+---------+------------+-------+--------+---------+
|ORD0006745| CUST005936|2026-07-14 06:31:37|   placed|    12390.52|  South|   Books|   WH-HYD|
|ORD0032675| CUST002305|2026-04-05 09:43:12|delivered|    14664.64|Central|  Sports|   WH-BLR|
|ORD0018160| CUST003165|2026-01-03 03:46:10|  shipped|    11196.85|  South|   Books|   WH-PUN|
|ORD0024150| CUST009112|2026-05-30 23:19:05|     NULL|     5709.42|  South|  Beauty|   WH-HYD|
|ORD0046507| CUST009632|2026-03-30 05:11:07|   placed|     3831.66|  North| Apparel|   WH-DEL|
+----------+-----------+-------------------+---------+------------+-------+--------+---------+
only showing top 5 rows


In [0]:
# adding metadata
from pyspark.sql.functions import current_timestamp, lit

orders_df = (
    orders_df
    .withColumn("landing_timestamp", current_timestamp())
    .withColumn("source_file_name", lit("orders.csv"))
)
orders_df.show()

+----------+-----------+-------------------+---------+------------+-------+-----------+---------+--------------------+----------------+
|  order_id|customer_id|         order_date|   status|total_amount| region|   category|warehouse|   landing_timestamp|source_file_name|
+----------+-----------+-------------------+---------+------------+-------+-----------+---------+--------------------+----------------+
|ORD0006745| CUST005936|2026-07-14 06:31:37|   placed|    12390.52|  South|      Books|   WH-HYD|2026-08-08 17:51:...|      orders.csv|
|ORD0032675| CUST002305|2026-04-05 09:43:12|delivered|    14664.64|Central|     Sports|   WH-BLR|2026-08-08 17:51:...|      orders.csv|
|ORD0018160| CUST003165|2026-01-03 03:46:10|  shipped|    11196.85|  South|      Books|   WH-PUN|2026-08-08 17:51:...|      orders.csv|
|ORD0024150| CUST009112|2026-05-30 23:19:05|     NULL|     5709.42|  South|     Beauty|   WH-HYD|2026-08-08 17:51:...|      orders.csv|
|ORD0046507| CUST009632|2026-03-30 05:11:07|   p

In [0]:
# saving as delta table
orders_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("ecommerce_landing.orders")

In [0]:
# checking orders table
landing_orders = spark.table("ecommerce_landing.orders")
landing_orders.show(5)

+----------+-----------+-------------------+---------+------------+-------+--------+---------+--------------------+----------------+
|  order_id|customer_id|         order_date|   status|total_amount| region|category|warehouse|   landing_timestamp|source_file_name|
+----------+-----------+-------------------+---------+------------+-------+--------+---------+--------------------+----------------+
|ORD0006745| CUST005936|2026-07-14 06:31:37|   placed|    12390.52|  South|   Books|   WH-HYD|2026-08-08 17:51:...|      orders.csv|
|ORD0032675| CUST002305|2026-04-05 09:43:12|delivered|    14664.64|Central|  Sports|   WH-BLR|2026-08-08 17:51:...|      orders.csv|
|ORD0018160| CUST003165|2026-01-03 03:46:10|  shipped|    11196.85|  South|   Books|   WH-PUN|2026-08-08 17:51:...|      orders.csv|
|ORD0024150| CUST009112|2026-05-30 23:19:05|     NULL|     5709.42|  South|  Beauty|   WH-HYD|2026-08-08 17:51:...|      orders.csv|
|ORD0046507| CUST009632|2026-03-30 05:11:07|   placed|     3831.66|  

In [0]:
# Read order_items.csv
order_items_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(order_items_file)
)
order_items_df.show(5)

+-------------+----------+--------+--------+----------+
|order_item_id|  order_id|     sku|quantity|unit_price|
+-------------+----------+--------+--------+----------+
|   OI00000001|ORD0043145|SKU02732|       7|     765.2|
|   OI00000002|ORD0033708|SKU00762|       2|   1767.12|
|   OI00000003|ORD0031641|SKU02028|       1|   2898.29|
|   OI00000004|ORD0028894|SKU04354|      10|   2578.68|
|   OI00000005|ORD0002016|SKU03359|       9|    379.12|
+-------------+----------+--------+--------+----------+
only showing top 5 rows


In [0]:
# adding metadata
order_items_df = (
    order_items_df
    .withColumn("landing_timestamp", F.current_timestamp())
    .withColumn("source_file_name", F.lit("order_items.csv"))
)

In [0]:
# saving as delta table
order_items_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("ecommerce_landing.order_items")

In [0]:
# checking order_items table
landing_order_items = spark.table("ecommerce_landing.order_items")
landing_order_items.show(5)

+-------------+----------+--------+--------+----------+--------------------+----------------+
|order_item_id|  order_id|     sku|quantity|unit_price|   landing_timestamp|source_file_name|
+-------------+----------+--------+--------+----------+--------------------+----------------+
|   OI00000001|ORD0043145|SKU02732|       7|     765.2|2026-08-08 17:52:...| order_items.csv|
|   OI00000002|ORD0033708|SKU00762|       2|   1767.12|2026-08-08 17:52:...| order_items.csv|
|   OI00000003|ORD0031641|SKU02028|       1|   2898.29|2026-08-08 17:52:...| order_items.csv|
|   OI00000004|ORD0028894|SKU04354|      10|   2578.68|2026-08-08 17:52:...| order_items.csv|
|   OI00000005|ORD0002016|SKU03359|       9|    379.12|2026-08-08 17:52:...| order_items.csv|
+-------------+----------+--------+--------+----------+--------------------+----------------+
only showing top 5 rows


In [0]:
# read customers.csv
customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(customers_file)
)
customers_df.show(5)

+-----------+----------+---------+--------------------+-------+-----------+
|customer_id|first_name|last_name|               email| region|signup_date|
+-----------+----------+---------+--------------------+-------+-----------+
| CUST000001|    Aditya|     Bhat|aditya.bhat1@exam...|  North| 2026-12-05|
| CUST000002|     Ayaan|     Iyer|ayaan.iyer2@examp...|   West| 2026-08-08|
| CUST000003|     Arjun|    Joshi|arjun.joshi3@exam...|Central| 2027-08-14|
| CUST000004|     Ayaan|      Das|ayaan.das4@exampl...|   West| 2027-02-21|
| CUST000005|    Vihaan|    Verma|vihaan.verma5@exa...|   West| 2026-12-29|
+-----------+----------+---------+--------------------+-------+-----------+
only showing top 5 rows


In [0]:
# adding metadata
customers_df = (
    customers_df
    .withColumn("landing_timestamp", F.current_timestamp())
    .withColumn("source_file_name", F.lit("customers.csv"))
)

In [0]:
# saving as delta table
customers_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("ecommerce_landing.customers")

In [0]:
# checking the customers table
landing_customers = spark.table("ecommerce_landing.customers")
landing_customers.show(5)

+-----------+----------+---------+--------------------+-------+-----------+--------------------+----------------+
|customer_id|first_name|last_name|               email| region|signup_date|   landing_timestamp|source_file_name|
+-----------+----------+---------+--------------------+-------+-----------+--------------------+----------------+
| CUST000001|    Aditya|     Bhat|aditya.bhat1@exam...|  North| 2026-12-05|2026-08-08 17:52:...|   customers.csv|
| CUST000002|     Ayaan|     Iyer|ayaan.iyer2@examp...|   West| 2026-08-08|2026-08-08 17:52:...|   customers.csv|
| CUST000003|     Arjun|    Joshi|arjun.joshi3@exam...|Central| 2027-08-14|2026-08-08 17:52:...|   customers.csv|
| CUST000004|     Ayaan|      Das|ayaan.das4@exampl...|   West| 2027-02-21|2026-08-08 17:52:...|   customers.csv|
| CUST000005|    Vihaan|    Verma|vihaan.verma5@exa...|   West| 2026-12-29|2026-08-08 17:52:...|   customers.csv|
+-----------+----------+---------+--------------------+-------+-----------+-------------

In [0]:
# read inventory.csv
inventory_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(inventory_file)
)
inventory_df.show(5)

+--------+------------+-----------+---------+--------------+-------------+
|     sku|product_name|   category|warehouse|stock_quantity|reorder_level|
+--------+------------+-----------+---------+--------------+-------------+
|SKU00001|   Product 1|     Beauty|   WH-BLR|           202|           15|
|SKU00002|   Product 2|    Apparel|   WH-HYD|           416|           13|
|SKU00003|   Product 3|Electronics|   WH-DEL|           355|           39|
|SKU00004|   Product 4|     Sports|   WH-BLR|           140|           15|
|SKU00005|   Product 5|     Beauty|   WH-MUM|           463|           36|
+--------+------------+-----------+---------+--------------+-------------+
only showing top 5 rows


In [0]:
# adding metadata
inventory_df = (
    inventory_df
    .withColumn("landing_timestamp", F.current_timestamp())
    .withColumn("source_file_name", F.lit("inventory.csv"))
)

In [0]:
# saving as delta table 
inventory_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("ecommerce_landing.inventory")

In [0]:
# checking inventory table
landing_inventory = spark.table("ecommerce_landing.inventory")
landing_inventory.show(5)

+--------+------------+-----------+---------+--------------+-------------+--------------------+----------------+
|     sku|product_name|   category|warehouse|stock_quantity|reorder_level|   landing_timestamp|source_file_name|
+--------+------------+-----------+---------+--------------+-------------+--------------------+----------------+
|SKU00001|   Product 1|     Beauty|   WH-BLR|           202|           15|2026-08-07 20:21:...|   inventory.csv|
|SKU00002|   Product 2|    Apparel|   WH-HYD|           416|           13|2026-08-07 20:21:...|   inventory.csv|
|SKU00003|   Product 3|Electronics|   WH-DEL|           355|           39|2026-08-07 20:21:...|   inventory.csv|
|SKU00004|   Product 4|     Sports|   WH-BLR|           140|           15|2026-08-07 20:21:...|   inventory.csv|
|SKU00005|   Product 5|     Beauty|   WH-MUM|           463|           36|2026-08-07 20:21:...|   inventory.csv|
+--------+------------+-----------+---------+--------------+-------------+--------------------+-

### 02 - Bronze Layer

### Objective

Read data from the Landing layer.

Keep all business columns as STRING.

Add Bronze metadata.

Store as Delta tables partitioned by load_date.

In [0]:
# Read orders table from Landing
orders_bronze = spark.table("ecommerce_landing.orders")
orders_bronze.show(5)

+----------+-----------+-------------------+---------+------------+-------+--------+---------+--------------------+----------------+
|  order_id|customer_id|         order_date|   status|total_amount| region|category|warehouse|   landing_timestamp|source_file_name|
+----------+-----------+-------------------+---------+------------+-------+--------+---------+--------------------+----------------+
|ORD0006745| CUST005936|2026-07-14 06:31:37|   placed|    12390.52|  South|   Books|   WH-HYD|2026-08-08 17:51:...|      orders.csv|
|ORD0032675| CUST002305|2026-04-05 09:43:12|delivered|    14664.64|Central|  Sports|   WH-BLR|2026-08-08 17:51:...|      orders.csv|
|ORD0018160| CUST003165|2026-01-03 03:46:10|  shipped|    11196.85|  South|   Books|   WH-PUN|2026-08-08 17:51:...|      orders.csv|
|ORD0024150| CUST009112|2026-05-30 23:19:05|     NULL|     5709.42|  South|  Beauty|   WH-HYD|2026-08-08 17:51:...|      orders.csv|
|ORD0046507| CUST009632|2026-03-30 05:11:07|   placed|     3831.66|  

In [0]:
# Add Bronze metadata
orders_bronze = (
    orders_bronze
    .withColumn("bronze_ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

In [0]:
# Save the Bronze table
orders_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("load_date") \
    .saveAsTable("ecommerce_bronze.orders")

In [0]:
# checking bronze_orders
bronze_orders = spark.table("ecommerce_bronze.orders")
bronze_orders.show(5)
print("Total rows:", bronze_orders.count())

+----------+-----------+-------------------+---------+------------+-------+--------+---------+--------------------+----------------+--------------------------+----------+
|  order_id|customer_id|         order_date|   status|total_amount| region|category|warehouse|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+----------+-----------+-------------------+---------+------------+-------+--------+---------+--------------------+----------------+--------------------------+----------+
|ORD0006745| CUST005936|2026-07-14 06:31:37|   placed|    12390.52|  South|   Books|   WH-HYD|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD0032675| CUST002305|2026-04-05 09:43:12|delivered|    14664.64|Central|  Sports|   WH-BLR|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD0018160| CUST003165|2026-01-03 03:46:10|  shipped|    11196.85|  South|   Books|   WH-PUN|2026-08-08 04:17:...|      orders.csv|      2026-08

In [0]:
# read order items table from landing
orders_items_bronze=spark.table("ecommerce_landing.order_items")

orders_items_bronze.show(5)

+-------------+----------+--------+--------+----------+--------------------+----------------+
|order_item_id|  order_id|     sku|quantity|unit_price|   landing_timestamp|source_file_name|
+-------------+----------+--------+--------+----------+--------------------+----------------+
|   OI00000001|ORD0043145|SKU02732|       7|     765.2|2026-08-08 17:52:...| order_items.csv|
|   OI00000002|ORD0033708|SKU00762|       2|   1767.12|2026-08-08 17:52:...| order_items.csv|
|   OI00000003|ORD0031641|SKU02028|       1|   2898.29|2026-08-08 17:52:...| order_items.csv|
|   OI00000004|ORD0028894|SKU04354|      10|   2578.68|2026-08-08 17:52:...| order_items.csv|
|   OI00000005|ORD0002016|SKU03359|       9|    379.12|2026-08-08 17:52:...| order_items.csv|
+-------------+----------+--------+--------+----------+--------------------+----------------+
only showing top 5 rows


In [0]:
# adding bronze metadata
orders_items_bronze = (
    orders_items_bronze
    .withColumn("bronze_ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

In [0]:
# save the bronze table
orders_items_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .partitionBy("load_date") \
    .saveAsTable("ecommerce_bronze.order_items")


In [0]:
# checking bronze_order_items
bronze_order_items = spark.table("ecommerce_bronze.order_items")
bronze_order_items.show(5)
print("Total rows:", bronze_order_items.count())

+-------------+----------+--------+--------+----------+--------------------+----------------+--------------------------+----------+
|order_item_id|  order_id|     sku|quantity|unit_price|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+-------------+----------+--------+--------+----------+--------------------+----------------+--------------------------+----------+
|   OI00000001|ORD0043145|SKU02732|       7|     765.2|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|
|   OI00000002|ORD0033708|SKU00762|       2|   1767.12|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|
|   OI00000003|ORD0031641|SKU02028|       1|   2898.29|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|
|   OI00000004|ORD0028894|SKU04354|      10|   2578.68|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|
|   OI00000005|ORD0002016|SKU03359|       9|    379.12|2026-08-08 04:17:...|

In [0]:
# read customers table from landing
customers_bronze=spark.table("ecommerce_landing.customers")
customers_bronze.show(5)

+-----------+----------+---------+--------------------+-------+-----------+--------------------+----------------+
|customer_id|first_name|last_name|               email| region|signup_date|   landing_timestamp|source_file_name|
+-----------+----------+---------+--------------------+-------+-----------+--------------------+----------------+
| CUST000001|    Aditya|     Bhat|aditya.bhat1@exam...|  North| 2026-12-05|2026-08-08 17:52:...|   customers.csv|
| CUST000002|     Ayaan|     Iyer|ayaan.iyer2@examp...|   West| 2026-08-08|2026-08-08 17:52:...|   customers.csv|
| CUST000003|     Arjun|    Joshi|arjun.joshi3@exam...|Central| 2027-08-14|2026-08-08 17:52:...|   customers.csv|
| CUST000004|     Ayaan|      Das|ayaan.das4@exampl...|   West| 2027-02-21|2026-08-08 17:52:...|   customers.csv|
| CUST000005|    Vihaan|    Verma|vihaan.verma5@exa...|   West| 2026-12-29|2026-08-08 17:52:...|   customers.csv|
+-----------+----------+---------+--------------------+-------+-----------+-------------

In [0]:
# adding bronze metadata
customers_bronze = (
    customers_bronze
    .withColumn("bronze_ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

In [0]:
# save the bronze table
customers_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .partitionBy("load_date") \
    .saveAsTable("ecommerce_bronze.customers")

In [0]:
# checking bronze_customers
bronze_customers = spark.table("ecommerce_bronze.customers")
bronze_customers.show(5)
print("Total rows:", bronze_customers.count())

+-----------+----------+---------+--------------------+-------+-----------+--------------------+----------------+--------------------------+----------+
|customer_id|first_name|last_name|               email| region|signup_date|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+-----------+----------+---------+--------------------+-------+-----------+--------------------+----------------+--------------------------+----------+
| CUST000001|    Aditya|     Bhat|aditya.bhat1@exam...|  North| 2026-12-05|2026-08-07 21:06:...|   customers.csv|      2026-08-08 17:52:...|2026-08-08|
| CUST000002|     Ayaan|     Iyer|ayaan.iyer2@examp...|   West| 2026-08-08|2026-08-07 21:06:...|   customers.csv|      2026-08-08 17:52:...|2026-08-08|
| CUST000003|     Arjun|    Joshi|arjun.joshi3@exam...|Central| 2027-08-14|2026-08-07 21:06:...|   customers.csv|      2026-08-08 17:52:...|2026-08-08|
| CUST000004|     Ayaan|      Das|ayaan.das4@exampl...|   West| 2027-02-21|2026-08-07 21

In [0]:
# reading inventory from landing table
inventory_bronze=spark.table("ecommerce_landing.inventory")
inventory_bronze.show(5)

+--------+------------+-----------+---------+--------------+-------------+--------------------+----------------+
|     sku|product_name|   category|warehouse|stock_quantity|reorder_level|   landing_timestamp|source_file_name|
+--------+------------+-----------+---------+--------------+-------------+--------------------+----------------+
|SKU00001|   Product 1|     Beauty|   WH-BLR|           202|           15|2026-08-07 20:21:...|   inventory.csv|
|SKU00002|   Product 2|    Apparel|   WH-HYD|           416|           13|2026-08-07 20:21:...|   inventory.csv|
|SKU00003|   Product 3|Electronics|   WH-DEL|           355|           39|2026-08-07 20:21:...|   inventory.csv|
|SKU00004|   Product 4|     Sports|   WH-BLR|           140|           15|2026-08-07 20:21:...|   inventory.csv|
|SKU00005|   Product 5|     Beauty|   WH-MUM|           463|           36|2026-08-07 20:21:...|   inventory.csv|
+--------+------------+-----------+---------+--------------+-------------+--------------------+-

In [0]:
# adding bronze metadata
inventory_bronze = (
    inventory_bronze
    .withColumn("bronze_ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

In [0]:
# save the bronze table
inventory_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .partitionBy("load_date") \
    .saveAsTable("ecommerce_bronze.inventory")


In [0]:
# checking bronze_inventory
bronze_inventory = spark.table("ecommerce_bronze.inventory")
bronze_inventory.show(5)
print("Total rows:", bronze_inventory.count())

+--------+------------+-----------+---------+--------------+-------------+--------------------+----------------+--------------------------+----------+
|     sku|product_name|   category|warehouse|stock_quantity|reorder_level|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+--------+------------+-----------+---------+--------------+-------------+--------------------+----------------+--------------------------+----------+
|SKU00001|   Product 1|     Beauty|   WH-BLR|           202|           15|2026-08-06 13:47:...|   inventory.csv|      2026-08-08 17:52:...|2026-08-08|
|SKU00002|   Product 2|    Apparel|   WH-HYD|           416|           13|2026-08-06 13:47:...|   inventory.csv|      2026-08-08 17:52:...|2026-08-08|
|SKU00003|   Product 3|Electronics|   WH-DEL|           355|           39|2026-08-06 13:47:...|   inventory.csv|      2026-08-08 17:52:...|2026-08-08|
|SKU00004|   Product 4|     Sports|   WH-BLR|           140|           15|2026-08-06 13:47:...

### 03 - Silver Layer

### Objective
Remove duplicate records.

Convert data types.

Apply data quality rules.

Create quarantine tables.

Build clean Silver tables.

In [0]:
# read orders table from bronze layer
orders_silver = spark.table("ecommerce_bronze.orders")
orders_silver.show(5)

+----------+-----------+-------------------+---------+------------+-------+--------+---------+--------------------+----------------+--------------------------+----------+
|  order_id|customer_id|         order_date|   status|total_amount| region|category|warehouse|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+----------+-----------+-------------------+---------+------------+-------+--------+---------+--------------------+----------------+--------------------------+----------+
|ORD0006745| CUST005936|2026-07-14 06:31:37|   placed|    12390.52|  South|   Books|   WH-HYD|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD0032675| CUST002305|2026-04-05 09:43:12|delivered|    14664.64|Central|  Sports|   WH-BLR|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD0018160| CUST003165|2026-01-03 03:46:10|  shipped|    11196.85|  South|   Books|   WH-PUN|2026-08-08 04:17:...|      orders.csv|      2026-08

In [0]:
# Create a window for duplicate orders
orders_window = (
    Window
    .partitionBy("order_id")
    .orderBy(F.col("bronze_ingestion_timestamp").desc())
)

In [0]:
# assign row Numbers
orders_silver = (
    orders_silver
    .withColumn(
        "row_number",
        F.row_number().over(orders_window)
    )
)
orders_silver.show(10)

+----------+-----------+-------------------+---------+------------+------+-----------+---------+--------------------+----------------+--------------------------+----------+----------+
|  order_id|customer_id|         order_date|   status|total_amount|region|   category|warehouse|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|row_number|
+----------+-----------+-------------------+---------+------------+------+-----------+---------+--------------------+----------------+--------------------------+----------+----------+
|ORD0000001| CUST003875|2026-04-27 09:31:33|delivered|     8792.19| North|Electronics|   WH-HYD|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|         1|
|ORD0000001| CUST003875|2026-04-27 09:31:33|delivered|     8792.19| North|Electronics|   WH-HYD|2026-08-07 21:06:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|         2|
|ORD0000001| CUST003875|2026-04-27 09:31:33|delivered|     8792.19| North|Electr

In [0]:
# remove duplicates
orders_silver = (
    orders_silver
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)
orders_silver.show(5)

+----------+-----------+-------------------+---------+------------+-------+-----------+---------+--------------------+----------------+--------------------------+----------+
|  order_id|customer_id|         order_date|   status|total_amount| region|   category|warehouse|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+----------+-----------+-------------------+---------+------------+-------+-----------+---------+--------------------+----------------+--------------------------+----------+
|ORD0000001| CUST003875|2026-04-27 09:31:33|delivered|     8792.19|  North|Electronics|   WH-HYD|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD0000002| CUST000559|2026-06-23 16:56:52|delivered|     8230.35|  North|     Sports|   WH-BLR|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD0000003| CUST002376|2026-02-24 23:24:27|  shipped|    10747.99|   East|      Books|   WH-PUN|2026-08-08 04:17:...|      orders

In [0]:
#type cast orders
orders_silver = (
    orders_silver
    .withColumn(
        "order_date",
        F.to_date(F.col("order_date"))
    )
    .withColumn(
        "total_amount",
        F.col("total_amount").cast("double")
    )
    .withColumn(
        "status",
        F.lower(F.col("status"))
    )
)

In [0]:
#Create valid orders
valid_orders = orders_silver.filter(
    (F.col("order_id").isNotNull()) &
    (F.col("customer_id").isNotNull()) &
    (F.col("total_amount") > 0) &
    (F.col("status").isin( "placed","shipped","delivered","cancelled"))
)
valid_orders.show(5)
print("Valid Orders:", valid_orders.count())

+----------+-----------+----------+---------+------------+-------+-----------+---------+--------------------+----------------+--------------------------+----------+
|  order_id|customer_id|order_date|   status|total_amount| region|   category|warehouse|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+----------+-----------+----------+---------+------------+-------+-----------+---------+--------------------+----------------+--------------------------+----------+
|ORD0000001| CUST003875|2026-04-27|delivered|     8792.19|  North|Electronics|   WH-HYD|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD0000002| CUST000559|2026-06-23|delivered|     8230.35|  North|     Sports|   WH-BLR|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD0000003| CUST002376|2026-02-24|  shipped|    10747.99|   East|      Books|   WH-PUN|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD000000

In [0]:
# Create the Quarantine table
orders_quarantine = (
    orders_silver.filter(
        (F.col("customer_id").isNull()) |
        (F.col("total_amount") <= 0) |
        (~F.col("status").isin(
            "placed",
            "shipped",
            "delivered",
            "cancelled"
        ))
    )
)

In [0]:
# quarantine reason
orders_quarantine = orders_quarantine.withColumn(
    "quarantine_reason",
    F.when(
        F.col("customer_id").isNull(),
        "Customer ID is NULL"
    ).when(
        F.col("total_amount") <= 0,
        "Invalid Total Amount"
    ).when(
        ~F.col("status").isin(
            "placed",
            "shipped",
            "delivered",
            "cancelled"
        ),
        "Invalid Status"
    )
)

In [0]:
# checking the quarantine table and reason showing or not
orders_quarantine.show(5)
print("Orders Quarantine:", orders_quarantine.count())

+----------+-----------+----------+----------+------------+------+--------+---------+--------------------+----------------+--------------------------+----------+-----------------+
|  order_id|customer_id|order_date|    status|total_amount|region|category|warehouse|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|quarantine_reason|
+----------+-----------+----------+----------+------------+------+--------+---------+--------------------+----------------+--------------------------+----------+-----------------+
|ORD0000010| CUST001832|2026-04-24|in_transit|    12607.14| North|    Toys|   WH-HYD|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|   Invalid Status|
|ORD0000012| CUST009202|2026-03-03|   returnd|     9236.26| South|    Home|   WH-HYD|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|   Invalid Status|
|ORD0000018| CUST002181|2026-05-09|       n/a|      6019.9|  West| Apparel|   WH-DEL|2026-08-08 04:1

In [0]:
# Save Orders and quarantine in delta Tables
valid_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_silver.orders")

orders_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_silver.orders_quarantine")

In [0]:
# checking the tables
spark.table("ecommerce_silver.orders").show(5)
spark.table("ecommerce_silver.orders_quarantine").show(5)

+----------+-----------+----------+---------+------------+-------+-----------+---------+--------------------+----------------+--------------------------+----------+
|  order_id|customer_id|order_date|   status|total_amount| region|   category|warehouse|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+----------+-----------+----------+---------+------------+-------+-----------+---------+--------------------+----------------+--------------------------+----------+
|ORD0000001| CUST003875|2026-04-27|delivered|     8792.19|  North|Electronics|   WH-HYD|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD0000002| CUST000559|2026-06-23|delivered|     8230.35|  North|     Sports|   WH-BLR|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD0000003| CUST002376|2026-02-24|  shipped|    10747.99|   East|      Books|   WH-PUN|2026-08-08 04:17:...|      orders.csv|      2026-08-08 17:52:...|2026-08-08|
|ORD000000

In [0]:
# read order_items table from bronze layer
order_items_silver = spark.table("ecommerce_bronze.order_items")

In [0]:
# create window for order_items
order_items_window =( 
        Window
        .partitionBy("order_item_id") \
        .orderBy(F.col("bronze_ingestion_timestamp").desc()))


In [0]:
# assign row numbers
order_items_silver = (
    order_items_silver
    .withColumn(
        "row_number",
        F.row_number().over(order_items_window)
    )
)
order_items_silver.show(10)

+-------------+----------+--------+--------+----------+--------------------+----------------+--------------------------+----------+----------+
|order_item_id|  order_id|     sku|quantity|unit_price|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|row_number|
+-------------+----------+--------+--------+----------+--------------------+----------------+--------------------------+----------+----------+
|   OI00000001|ORD0043145|SKU02732|       7|     765.2|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|         1|
|   OI00000001|ORD0043145|SKU02732|       7|     765.2|2026-08-07 21:06:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|         2|
|   OI00000001|ORD0043145|SKU02732|       7|     765.2|2026-08-07 20:53:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|         3|
|   OI00000001|ORD0043145|SKU02732|       7|     765.2|2026-08-07 20:21:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|         4|

In [0]:
# remove duplicates
order_items_silver=(
    order_items_silver
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)
order_items_silver.show(5)

+-------------+----------+--------+--------+----------+--------------------+----------------+--------------------------+----------+
|order_item_id|  order_id|     sku|quantity|unit_price|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+-------------+----------+--------+--------+----------+--------------------+----------------+--------------------------+----------+
|   OI00000001|ORD0043145|SKU02732|       7|     765.2|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|
|   OI00000002|ORD0033708|SKU00762|       2|   1767.12|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|
|   OI00000003|ORD0031641|SKU02028|       1|   2898.29|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|
|   OI00000004|ORD0028894|SKU04354|      10|   2578.68|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|
|   OI00000005|ORD0002016|SKU03359|       9|    379.12|2026-08-08 04:17:...|

In [0]:
# typecast order_items
order_items_silver = (
    order_items_silver
    .withColumn("quantity",F.col("quantity").cast("int"))
    .withColumn("unit_price",F.col("unit_price").cast("double"))
)

In [0]:
# validate order_items
valid_order_items = (
    order_items_silver.filter(
        (F.col("quantity").isNotNull()) &
        (F.col("quantity")>0) &
        (F.col("unit_price").isNotNull()) &
        (F.col("unit_price")>0)
    )
)

In [0]:
# Quarantine table order_items
order_items_quarantine = (
    order_items_silver.filter(
        (F.col("quantity").isNull()) |
        (F.col("quantity")<=0) |
        (F.col("unit_price").isNull()) |
        (F.col("unit_price")<=0)
    )
)

In [0]:
# quarantine reason of order_items
order_items_quarantine = (
    order_items_quarantine.withColumn(
        "quarantine_reason",
        F.when(
            F.col("quantity").isNull(),
            "Quantity NULL"
        ).when(
            F.col("quantity")<=0,
            "Quantity <=0"
        ).when(
            F.col("unit_price").isNull(),
            "Unit Price NULL"
        ).otherwise(
            "Unit Price <=0"
        )
    )
)

In [0]:
# saving order_items and order_items_quarantine in delta tables
valid_order_items.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("ecommerce_silver.order_items")

order_items_quarantine.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("ecommerce_silver.order_items_quarantine")

In [0]:
# checking the DataFrames
spark.table("ecommerce_silver.order_items").show(5)
spark.table("ecommerce_silver.order_items_quarantine").show(5)

+-------------+----------+--------+--------+----------+--------------------+----------------+--------------------------+----------+----------+
|order_item_id|  order_id|     sku|quantity|unit_price|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|row_number|
+-------------+----------+--------+--------+----------+--------------------+----------------+--------------------------+----------+----------+
|   OI00000001|ORD0043145|SKU02732|       7|     765.2|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|      NULL|
|   OI00000002|ORD0033708|SKU00762|       2|   1767.12|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|      NULL|
|   OI00000003|ORD0031641|SKU02028|       1|   2898.29|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|      NULL|
|   OI00000004|ORD0028894|SKU04354|      10|   2578.68|2026-08-08 04:17:...| order_items.csv|      2026-08-08 17:52:...|2026-08-08|      NULL|

In [0]:
# Read Customers from Bronze 
customers_silver = spark.table("ecommerce_bronze.customers")

In [0]:
# typecast customers
customers_silver = (
    customers_silver
    .withColumn(
        "signup_date",
        F.to_date(F.col("signup_date"))
    )
)

In [0]:
# valid customers
valid_customers = customers_silver.filter(
    F.col("email").rlike(
        "^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"
    )
)

In [0]:
# save customers to delta table
customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_silver.customers")

In [0]:
# remove duplicate customer id
customers_window = Window.partitionBy("customer_id") \
    .orderBy(F.col("bronze_ingestion_timestamp").desc())

customers_silver = (
    customers_silver
    .withColumn("row_number", F.row_number().over(customers_window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

In [0]:
# scd type-1( if customer present update or overwrite with new record ,if not present insert new record)
from delta.tables import DeltaTable

if not spark.catalog.tableExists("ecommerce_silver.customers"):

    customers_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce_silver.customers")

    print("First Run Completed")
else:

    target = DeltaTable.forName(
        spark,
        "ecommerce_silver.customers"
    )

    (
        target.alias("target")
        .merge(
            customers_silver.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdate(
            set={
                "first_name": "source.first_name",
                "last_name": "source.last_name",
                "email": "source.email",
                "region": "source.region",
                "signup_date": "source.signup_date",
                "landing_timestamp": "source.landing_timestamp",
                "source_file_name": "source.source_file_name",
                "bronze_ingestion_timestamp": "source.bronze_ingestion_timestamp",
                "load_date": "source.load_date"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("SCD Type-1 Merge Completed")

SCD Type-1 Merge Completed


In [0]:
# checking the table
spark.table("ecommerce_silver.customers").show(5)
spark.table("ecommerce_silver.customers").count()

+-----------+----------+---------+--------------------+-------+-----------+--------------------+----------------+--------------------------+----------+
|customer_id|first_name|last_name|               email| region|signup_date|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+-----------+----------+---------+--------------------+-------+-----------+--------------------+----------------+--------------------------+----------+
| CUST000001|    Aditya|     Bhat|aditya.bhat1@exam...|  North| 2026-12-05|2026-08-07 21:06:...|   customers.csv|      2026-08-08 17:52:...|2026-08-08|
| CUST000002|     Ayaan|     Iyer|ayaan.iyer2@examp...|   West| 2026-08-08|2026-08-07 21:06:...|   customers.csv|      2026-08-08 17:52:...|2026-08-08|
| CUST000003|     Arjun|    Joshi|arjun.joshi3@exam...|Central| 2027-08-14|2026-08-07 21:06:...|   customers.csv|      2026-08-08 17:52:...|2026-08-08|
| CUST000004|     Ayaan|      Das|ayaan.das4@exampl...|   West| 2027-02-21|2026-08-07 21

100000

In [0]:
# Read bronze inventory
inventory_silver = spark.table("ecommerce_bronze.inventory")

In [0]:
# type casting inventory
inventory_silver = (
    inventory_silver
    .withColumn(
        "stock_quantity",
        F.col("stock_quantity").cast("int")
    )
    .withColumn(
        "reorder_level",
        F.col("reorder_level").cast("int")
    )
)

In [0]:
# validate inventory
valid_inventory = inventory_silver.filter(
    F.col("stock_quantity").isNotNull()
)

In [0]:
#  advisory warning
negative_stock_count = valid_inventory.filter(
    F.col("stock_quantity") < 0
).count()

print("Warning - Negative Stock Records:", negative_stock_count)

Warning - Negative Stock Records: 0


In [0]:
# save the inventory to delta table
inventory_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_silver.inventory")

In [0]:
# checking inventory table
spark.table("ecommerce_silver.inventory").show(5)
spark.table("ecommerce_silver.inventory").count()

+--------+------------+-----------+---------+--------------+-------------+--------------------+----------------+--------------------------+----------+
|     sku|product_name|   category|warehouse|stock_quantity|reorder_level|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|
+--------+------------+-----------+---------+--------------+-------------+--------------------+----------------+--------------------------+----------+
|SKU00001|   Product 1|     Beauty|   WH-BLR|           202|           15|2026-08-06 13:47:...|   inventory.csv|      2026-08-08 17:52:...|2026-08-08|
|SKU00002|   Product 2|    Apparel|   WH-HYD|           416|           13|2026-08-06 13:47:...|   inventory.csv|      2026-08-08 17:52:...|2026-08-08|
|SKU00003|   Product 3|Electronics|   WH-DEL|           355|           39|2026-08-06 13:47:...|   inventory.csv|      2026-08-08 17:52:...|2026-08-08|
|SKU00004|   Product 4|     Sports|   WH-BLR|           140|           15|2026-08-06 13:47:...

45000

## 04-Gold Layer

The Gold layer is the final layer of the Medallion Architecture. Here, the cleaned data from the Silver layer is transformed into business-friendly tables that can be used for reporting and analysis.

In this layer, I will create the following tables:

- Daily Revenue
- Fulfillment KPI
- Inventory Health
- Customer Lifetime Value (LTV)

These tables help in understanding sales performance, order fulfillment, inventory status, and customer behavior.

##Daily Revenue

In [0]:
# read silver orders table
orders_gold = spark.table("ecommerce_silver.orders")

In [0]:
# create aggregation
daily_revenue = (
    orders_gold
    .filter(F.col("status") != "cancelled")
    .groupBy(
        F.to_date("order_date").alias("order_date"),
        "region",
        "category"
    )
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.count("order_id").alias("order_count"),
        F.avg("total_amount").alias("avg_order_value")
    )
)

In [0]:
# save to the delta table
daily_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_gold.daily_revenue")

In [0]:
# checking daily revenue table
spark.table("ecommerce_gold.daily_revenue").show(5)
print(
    "Daily Revenue Rows:",
    spark.table("ecommerce_gold.daily_revenue").count()
)

+----------+-------+--------+------------------+-----------+-----------------+
|order_date| region|category|     total_revenue|order_count|  avg_order_value|
+----------+-------+--------+------------------+-----------+-----------------+
|2026-06-08|  South|  Beauty|           3812.05|          2|         1906.025|
|2026-02-04|  South|  Beauty|          23064.41|          4|        5766.1025|
|2026-02-14|  North|   Books|          30754.97|          6|5125.828333333334|
|2026-06-09|Central| Apparel|          15795.11|          4|        3948.7775|
|2026-06-05|   West| Apparel|30040.809999999998|          5|6008.161999999999|
+----------+-------+--------+------------------+-----------+-----------------+
only showing top 5 rows
Daily Revenue Rows: 7261


In [0]:
# daily revenue KPI summary

daily_revenue = spark.table("ecommerce_gold.daily_revenue")

total_revenue = daily_revenue.agg(
    F.sum("total_revenue")
).first()[0]

total_orders = daily_revenue.agg(
    F.sum("order_count")
).first()[0]

avg_order_value = daily_revenue.agg(
    F.avg("avg_order_value")
).first()[0]

print("Total Revenue:", total_revenue)
print("Total Orders:", total_orders)
print("Average Order Value:", avg_order_value)

print("\nTop Region:")
daily_revenue.groupBy("region") \
    .agg(F.sum("total_revenue").alias("revenue")) \
    .orderBy(F.desc("revenue")) \
    .show(5, False)

print("Top Category:")
daily_revenue.groupBy("category") \
    .agg(F.sum("total_revenue").alias("revenue")) \
    .orderBy(F.desc("revenue")) \
    .show(5, False)

Total Revenue: 231610971.4200002
Total Orders: 30559
Average Order Value: 7570.346080762669

Top Region:
+-------+--------------------+
|region |revenue             |
+-------+--------------------+
|East   |4.7361624519999914E7|
|Central|4.6324089230000004E7|
|South  |4.6183077200000025E7|
|North  |4.602747536999996E7 |
|West   |4.571470510000003E7 |
+-------+--------------------+

Top Category:
+--------+--------------------+
|category|revenue             |
+--------+--------------------+
|Home    |3.391284856000001E7 |
|Books   |3.3386165450000014E7|
|Sports  |3.337194833000005E7 |
|Apparel |3.286661648000004E7 |
|Beauty  |3.2855488279999986E7|
+--------+--------------------+
only showing top 5 rows


## Fulfillment KPI

In [0]:
# read silver orders table
orders_gold = spark.table("ecommerce_silver.orders")

In [0]:
# create fulfillment kpi
fulfillment_kpi = (
    orders_gold
    .groupBy(
        F.to_date("order_date").alias("order_date"),
        "warehouse",
        "region"
    )
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum(
            F.when(F.col("status") == "delivered", 1).otherwise(0)
        ).alias("delivered_count"),
        F.sum(
            F.when(F.col("status") == "cancelled", 1).otherwise(0)
        ).alias("cancelled_count"),
        F.sum(
            F.when(F.col("status") == "shipped", 1).otherwise(0)
        ).alias("shipped_count")
    )
)

In [0]:
# calculate kpi percentage
fulfillment_kpi = (
    fulfillment_kpi
    .withColumn(
        "delivery_rate_pct",
        F.round(
            (F.col("delivered_count") / F.col("total_orders")) * 100,
            2
        )
    )
    .withColumn(
        "cancellation_rate_pct",
        F.round(
            (F.col("cancelled_count") / F.col("total_orders")) * 100,
            2
        )
    )
    .withColumn(
        "shipment_rate_pct",
        F.round(
            (F.col("shipped_count") / F.col("total_orders")) * 100,
            2
        )
    )
)

In [0]:
# save to the delta table
fulfillment_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_gold.fulfillment_kpi")

In [0]:
spark.table("ecommerce_gold.fulfillment_kpi").show(5)
print(
    "fulfillment KPI count:",
    spark.table("ecommerce_gold.fulfillment_kpi").count()
)

+----------+---------+------+------------+---------------+---------------+-------------+-----------------+---------------------+-----------------+
|order_date|warehouse|region|total_orders|delivered_count|cancelled_count|shipped_count|delivery_rate_pct|cancellation_rate_pct|shipment_rate_pct|
+----------+---------+------+------------+---------------+---------------+-------------+-----------------+---------------------+-----------------+
|2026-03-23|   WH-PUN|  West|           9|              3|              3|            3|            33.33|                33.33|            33.33|
|2026-06-24|   WH-BLR| North|           7|              1|              2|            2|            14.29|                28.57|            28.57|
|2026-01-31|   WH-DEL|  West|          10|              2|              4|            2|             20.0|                 40.0|             20.0|
|2026-06-02|   WH-DEL|  West|          12|              2|              5|            3|            16.67|            

In [0]:
# fulfillment KPI summary
fulfillment_kpi = spark.table("ecommerce_gold.fulfillment_kpi")
print("Average Delivery Rate:")
fulfillment_kpi.agg(
    F.avg("delivery_rate_pct").alias("delivery_rate_pct")
).show()
print("Average Cancellation Rate:")
fulfillment_kpi.agg(
    F.avg("cancellation_rate_pct").alias("cancellation_rate_pct")
).show()
print("Average Shipment Rate:")
fulfillment_kpi.agg(
    F.avg("shipment_rate_pct").alias("shipment_rate_pct")
).show()
print("Best Warehouse:")
fulfillment_kpi.groupBy("warehouse") \
    .agg(
        F.avg("delivery_rate_pct").alias("delivery_rate_pct")
    ) \
    .orderBy(F.desc("delivery_rate_pct")) \
    .show(5, False)
print("Worst Warehouse:")
fulfillment_kpi.groupBy("warehouse") \
    .agg(
        F.avg("delivery_rate_pct").alias("delivery_rate_pct")
    ) \
    .orderBy("delivery_rate_pct") \
    .show(5, False)

Average Delivery Rate:
+------------------+
| delivery_rate_pct|
+------------------+
|25.058412367223163|
+------------------+

Average Cancellation Rate:
+---------------------+
|cancellation_rate_pct|
+---------------------+
|   24.817886949924226|
+---------------------+

Average Shipment Rate:
+-----------------+
|shipment_rate_pct|
+-----------------+
|25.27572268588782|
+-----------------+

Best Warehouse:
+---------+------------------+
|warehouse|delivery_rate_pct |
+---------+------------------+
|WH-PUN   |25.65172675521826 |
|WH-DEL   |25.326654028436092|
|WH-MUM   |24.94887203791478 |
|WH-BLR   |24.895384615384696|
|WH-HYD   |24.469677725118572|
+---------+------------------+

Worst Warehouse:
+---------+------------------+
|warehouse|delivery_rate_pct |
+---------+------------------+
|WH-HYD   |24.469677725118572|
|WH-BLR   |24.895384615384696|
|WH-MUM   |24.94887203791478 |
|WH-DEL   |25.326654028436092|
|WH-PUN   |25.65172675521826 |
+---------+------------------+



## Inventory Health

In [0]:
# read silver tables
orders_gold = spark.table("ecommerce_silver.orders")
order_items_gold = spark.table("ecommerce_silver.order_items")
inventory_gold = spark.table("ecommerce_silver.inventory")

In [0]:
# find the last order date
max_order_date = orders_gold.agg(
    F.max("order_date")
).first()[0]

In [0]:
# 30 days cutoff
cutoff_date = F.lit(max_order_date) - F.expr("INTERVAL 30 DAYS")

In [0]:
# join orders with order items
items_with_date = (
    order_items_gold
    .join(
        orders_gold.select("order_id", "order_date"),
        "order_id"
    )
)

In [0]:
# orders in last 30 days
recent_items = items_with_date.filter(
    F.col("order_date") >= cutoff_date
)

In [0]:
# calculating the demand
demand_30d = (
    recent_items
    .groupBy("sku")
    .agg(
        F.sum("quantity").alias("demand_30d")
    )
)

In [0]:
# join with inventory
inventory_health = (
    inventory_gold
    .join(
        demand_30d,
        "sku",
        "left"
    )
)

In [0]:
# replace null demand
inventory_health = inventory_health.withColumn(
    "demand_30d",
    F.coalesce(
        F.col("demand_30d"),
        F.lit(0)
    )
)

In [0]:
# stock status
inventory_health = (
    inventory_health
    .withColumn(
        "stock_status",
        F.when(
            F.col("stock_quantity") == 0,
            "stockout"
        )
        .when(
            F.col("stock_quantity") < F.col("reorder_level"),
            "below_reorder"
        )
        .when(
            F.col("stock_quantity") > F.col("reorder_level") * 5,
            "overstock"
        )
        .otherwise("healthy")
    )
)

In [0]:
# reorder flag
inventory_health = inventory_health.withColumn(
    "reorder_flag",
    F.col("stock_quantity") < F.col("reorder_level")
)

In [0]:
# save to the delta table
inventory_health.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_gold.inventory_health") 

In [0]:
# checking the table
spark.table("ecommerce_gold.inventory_health").show(5)
print(
    "Inventory Health Count:",
    spark.table("ecommerce_gold.inventory_health").count()
)

+--------+------------+-----------+---------+--------------+-------------+--------------------+----------------+--------------------------+----------+----------+------------+------------+
|     sku|product_name|   category|warehouse|stock_quantity|reorder_level|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|demand_30d|stock_status|reorder_flag|
+--------+------------+-----------+---------+--------------+-------------+--------------------+----------------+--------------------------+----------+----------+------------+------------+
|SKU00001|   Product 1|     Beauty|   WH-BLR|           202|           15|2026-08-06 13:47:...|   inventory.csv|      2026-08-08 17:52:...|2026-08-08|        35|   overstock|       false|
|SKU00002|   Product 2|    Apparel|   WH-HYD|           416|           13|2026-08-06 13:47:...|   inventory.csv|      2026-08-08 17:52:...|2026-08-08|         8|   overstock|       false|
|SKU00003|   Product 3|Electronics|   WH-DEL|           355|

In [0]:
# Inventory Health KPI Summary
inventory_health = spark.table("ecommerce_gold.inventory_health")
total_skus = inventory_health.select("sku").distinct().count()
stockout_count = inventory_health.filter(
    F.col("stock_status") == "stockout"
).count()
below_reorder_count = inventory_health.filter(
    F.col("stock_status") == "below_reorder"
).count()
overstock_count = inventory_health.filter(
    F.col("stock_status") == "overstock"
).count()
print("Total SKUs:", total_skus)
print("Stockout Count:", stockout_count)
print("Below Reorder Count:", below_reorder_count)
print("Overstock Count:", overstock_count)
print("\nStock Status Breakdown:")
inventory_health.groupBy("stock_status").count().show()

===== INVENTORY HEALTH KPI SUMMARY =====
Total SKUs: 5000
Stockout Count: 135
Below Reorder Count: 2745
Overstock Count: 30384

Stock Status Breakdown:
+-------------+-----+
| stock_status|count|
+-------------+-----+
|    overstock|30384|
|below_reorder| 2745|
|      healthy|11736|
|     stockout|  135|
+-------------+-----+



## Customer Lifetime Value (LTV)

In [0]:
# reading thetables of orders  and customers in ecommerce_silver
orders_gold = spark.table("ecommerce_silver.orders")
customers_gold = spark.table("ecommerce_silver.customers")

In [0]:
# Calculate customer lifetime spend, order frequency, and last order date
customer_orders = (
    orders_gold
    .filter(F.col("status") != "cancelled")
    .groupBy("customer_id")
    .agg(
        F.sum("total_amount").alias("lifetime_spend"),
        F.count("order_id").alias("order_frequency"),
        F.max("order_date").alias("last_order_date")
    )
)

In [0]:
# joining customer information with customer order metrics
customer_ltv = (
    customers_gold
    .join(customer_orders, "customer_id", "left")
)

In [0]:
# Calculating the maximum order date and customer LTV metrics
max_order_date = orders_gold.agg(
    F.max("order_date")
).first()[0]

customer_ltv = (
    customer_ltv
    .withColumn(
        "lifetime_spend",
        F.coalesce(F.col("lifetime_spend"), F.lit(0.0))
    )
    .withColumn(
        "order_frequency",
        F.coalesce(F.col("order_frequency"), F.lit(0))
    )
    .withColumn(
        "recency_days",
        F.datediff(
            F.lit(max_order_date),
            F.col("last_order_date")
        )
    )
)

In [0]:
# customer segmentation
customer_ltv = customer_ltv.withColumn(
    "segment",
    F.when(F.col("lifetime_spend") >= 50000, "VIP")
     .when(F.col("lifetime_spend") >= 20000, "High Value")
     .when(F.col("lifetime_spend") >= 5000, "Mid Value")
     .otherwise("Low Value")
)

In [0]:
# saving to delta table
customer_ltv.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_gold.customer_ltv")

In [0]:
# checking the gold table
spark.table("ecommerce_gold.customer_ltv").show(5)
print(
    "Customer LTV Count:",
    spark.table("ecommerce_gold.customer_ltv").count()
)

+-----------+----------+---------+--------------------+-------+-----------+--------------------+----------------+--------------------------+----------+------------------+---------------+---------------+------------+----------+
|customer_id|first_name|last_name|               email| region|signup_date|   landing_timestamp|source_file_name|bronze_ingestion_timestamp| load_date|    lifetime_spend|order_frequency|last_order_date|recency_days|   segment|
+-----------+----------+---------+--------------------+-------+-----------+--------------------+----------------+--------------------------+----------+------------------+---------------+---------------+------------+----------+
| CUST000001|    Aditya|     Bhat|aditya.bhat1@exam...|  North| 2026-12-05|2026-08-07 21:06:...|   customers.csv|      2026-08-08 17:52:...|2026-08-08|53717.159999999996|              8|     2026-07-03|          27|       VIP|
| CUST000002|     Ayaan|     Iyer|ayaan.iyer2@examp...|   West| 2026-08-08|2026-08-07 21:06:

In [0]:
# customer(ltv) KPI summary
customer_ltv = spark.table("ecommerce_gold.customer_ltv")
total_customers = customer_ltv.count()
active_customers = customer_ltv.filter(
    F.col("order_frequency") > 0
).count()
active_customer_pct = (
    active_customers / total_customers * 100
)
average_ltv = customer_ltv.agg(
    F.avg("lifetime_spend")
).first()[0]
print("Total Customers:", total_customers)
print("Active Customer %:", round(active_customer_pct, 2))
print("Average LTV:", round(average_ltv, 2))
print("\nSegment Breakdown:")
customer_ltv.groupBy("segment") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

Total Customers: 100000
Active Customer %: 95.17
Average LTV: 23161.1

Segment Breakdown:
+----------+-----+
|   segment|count|
+----------+-----+
|High Value|48020|
| Mid Value|35650|
| Low Value|10700|
|       VIP| 5630|
+----------+-----+



## 05- Reconciliation

Reconciliation is used to check the flow of data between the different layers of the pipeline. It helps verify row counts and shows how many records passed the data quality checks or were sent to quarantine.

In [0]:
# row counts orders
landing_orders = spark.table("ecommerce_landing.orders").count()
bronze_orders = spark.table("ecommerce_bronze.orders").count()
silver_orders = spark.table("ecommerce_silver.orders").count()

print("Orders")
print("Landing:", landing_orders)
print("Bronze:", bronze_orders)
print("Silver:", silver_orders)

Orders
Landing: 501500
Bronze: 501500
Silver: 40598


In [0]:
# row counts order_items
landing_items = spark.table("ecommerce_landing.order_items").count()
bronze_items = spark.table("ecommerce_bronze.order_items").count()
silver_items = spark.table("ecommerce_silver.order_items").count()

print("Order Items")
print("Landing:", landing_items)
print("Bronze:", bronze_items)
print("Silver:", silver_items)

Order Items
Landing: 2000000
Bronze: 2000000
Silver: 195025


In [0]:
# row counts customers
landing_customers = spark.table("ecommerce_landing.customers").count()
bronze_customers = spark.table("ecommerce_bronze.customers").count()
silver_customers = spark.table("ecommerce_silver.customers").count()

print("Customers")
print("Landing:", landing_customers)
print("Bronze:", bronze_customers)
print("Silver:", silver_customers)

Customers
Landing: 100000
Bronze: 100000
Silver: 100000


In [0]:
# row counts inventory
landing_inventory = spark.table("ecommerce_landing.inventory").count()
bronze_inventory = spark.table("ecommerce_bronze.inventory").count()
silver_inventory = spark.table("ecommerce_silver.inventory").count()

print("Inventory")
print("landing: ", landing_inventory)
print("bronze: ", bronze_inventory)
print("silver: ", silver_inventory)

Inventory
landing:  45000
bronze:  45000
silver:  45000


In [0]:
# creating reconciliation dataframe
row_counts_data = [
    ("orders", landing_orders, bronze_orders, silver_orders),
    ("order_items", landing_items, bronze_items, silver_items),
    ("customers", landing_customers, bronze_customers, silver_customers),
    ("inventory", landing_inventory, bronze_inventory, silver_inventory)
]
row_counts_df = spark.createDataFrame(
    row_counts_data,
    ["table_name", "landing_count", "bronze_count", "silver_count"]
).withColumn(
    "captured_at",
    F.current_timestamp()
)

In [0]:
# saving to delta table
row_counts_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_gold.reconciliation_row_counts")

In [0]:
# checking the total counts in landing,bronze and silver layer
row_counts_df.show()

+-----------+-------------+------------+------------+--------------------+
| table_name|landing_count|bronze_count|silver_count|         captured_at|
+-----------+-------------+------------+------------+--------------------+
|     orders|       501500|      501500|       40598|2026-08-08 19:54:...|
|order_items|      2000000|     2000000|      195025|2026-08-08 19:54:...|
|  customers|       100000|      100000|      100000|2026-08-08 19:54:...|
|  inventory|        45000|       45000|       45000|2026-08-08 19:54:...|
+-----------+-------------+------------+------------+--------------------+



## 06- Reconciliation DQ Summary

This table summarizes the data quality results for the orders and order_items tables. It shows the number of records entering Bronze, the records that passed into Silver, the records sent to quarantine, and the pass and quarantine rates.


In [0]:
# read the quarantined data of orders and order items
orders_quarantined = spark.table(
    "ecommerce_silver.orders_quarantine"
).count()
items_quarantined = spark.table(
    "ecommerce_silver.order_items_quarantine"
).count()

In [0]:
# Create DQ summary for orders and order_items
dq_data = [
    (
        "orders",
        bronze_orders,
        silver_orders,
        orders_quarantined,
        round(silver_orders / bronze_orders * 100, 2),
        round(orders_quarantined / bronze_orders * 100, 2)
    ),
    (
        "order_items",
        bronze_items,
        silver_items,
        items_quarantined,
        round(silver_items / bronze_items * 100, 2),
        round(items_quarantined / bronze_items * 100, 2)
    )
]
dq_df = spark.createDataFrame(
    dq_data,
    [
        "table_name",
        "bronze_row_count",
        "silver_row_count",
        "quarantined_rows",
        "pass_rate_pct",
        "quarantine_rate_pct"
    ]
).withColumn(
    "captured_at",
    F.current_timestamp()
)

In [0]:
# Save DQ summary to Gold
dq_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_gold.reconciliation_dq_summary"
    )

In [0]:
# checking the data in the dataframe
dq_df.show()

+-----------+----------------+----------------+----------------+-------------+-------------------+--------------------+
| table_name|bronze_row_count|silver_row_count|quarantined_rows|pass_rate_pct|quarantine_rate_pct|         captured_at|
+-----------+----------------+----------------+----------------+-------------+-------------------+--------------------+
|     orders|          501500|           40598|            8127|          8.1|               1.62|2026-08-08 19:54:...|
|order_items|         2000000|          195025|            4975|         9.75|               0.25|2026-08-08 19:54:...|
+-----------+----------------+----------------+----------------+-------------+-------------------+--------------------+



In [0]:
# Final Gold and Reconciliation Verification
print("gold tables")
spark.sql("SHOW TABLES IN ecommerce_gold").show(truncate=False)
print("reconciliation row counts")
spark.table(
    "ecommerce_gold.reconciliation_row_counts"
).show(truncate=False)
print("reconciliation dq summary")
spark.table(
    "ecommerce_gold.reconciliation_dq_summary"
).show(truncate=False)

gold tables
+--------------+-------------------------+-----------+
|database      |tableName                |isTemporary|
+--------------+-------------------------+-----------+
|ecommerce_gold|customer_ltv             |false      |
|ecommerce_gold|daily_revenue            |false      |
|ecommerce_gold|fulfillment_kpi          |false      |
|ecommerce_gold|inventory_health         |false      |
|ecommerce_gold|reconciliation_dq_summary|false      |
|ecommerce_gold|reconciliation_row_counts|false      |
+--------------+-------------------------+-----------+

reconciliation row counts
+-----------+-------------+------------+------------+-------------------------+
|table_name |landing_count|bronze_count|silver_count|captured_at              |
+-----------+-------------+------------+------------+-------------------------+
|orders     |501500       |501500      |40598       |2026-08-08 19:54:12.20897|
|order_items|2000000      |2000000     |195025      |2026-08-08 19:54:12.20897|
|customers 